In [1]:
import os
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma


c:\Users\Hp\OneDrive\Desktop\Rag\rag_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
current_dir = r"C:\Users\Hp\OneDrive\Desktop\Rag"
file_path = os.path.join(current_dir, "AMZN-Q4-2025-Earnings-Release.txt")
persistent_directory = os.path.join(current_dir, "db", "chroma_db")



In [3]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv

In [4]:
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

C:\Users\Hp\AppData\Local\Temp\ipykernel_10372\4145769993.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 334.85it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
# Check if the Chroma vector store already exists
if not os.path.exists(persistent_directory):
    print("Persistent directory does not exist. Initializing vector store...")

    # Ensure the text file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(
            
            f"The file {file_path} does not exist. Please check the path."
        )

    # Read the text content from the file
    loader = TextLoader(file_path, encoding="utf-8")
    documents = loader.load()

    # Split the document into chunks
    rec_char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=100)
    docs = rec_char_splitter.split_documents(documents)
    # Display information about the split documents
    print("\n--- Document Chunks Information ---")
    print(f"Number of document chunks: {len(docs)}")
    print(f"Sample chunk:\n{docs[0].page_content}\n")

    # Create embeddings
    print("\n--- Creating embeddings ---")
    print("\n--- Finished creating embeddings ---")

    # Create the vector store and persist it automatically
    print("\n--- Creating vector store ---")
    db = Chroma.from_documents(
        docs, embeddings, persist_directory=persistent_directory)
    print("\n--- Finished creating vector store ---")

else:
    print("Vector store already exists. No need to initialize.")

Vector store already exists. No need to initialize.


In [6]:
import os

from langchain_community.vectorstores import Chroma
 
# Load the existing vector store with the embedding function
db = Chroma(persist_directory=persistent_directory,
            embedding_function=embeddings)

# Define the user's question
query = "Net sales of north america" 

# Retrieve relevant documents based on the query
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)
relevant_docs = retriever.invoke(query)

# Display the relevant results with metadata
print("\n--- Relevant Documents ---")
for i, doc in enumerate(relevant_docs, 1):
    print(f"Document {i}:\n{doc.page_content}\n")
    if doc.metadata:
        print(f"Source: {doc.metadata.get('source', 'Unknown')}\n")

C:\Users\Hp\AppData\Local\Temp\ipykernel_10372\1587192282.py:6: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db = Chroma(persist_directory=persistent_directory,



--- Relevant Documents ---
Document 1:
Net income increased to $21.2 billion in the fourth quarter, or $1.95 per diluted share, compared with $20.0 billion, or
$1.86 per diluted share, in fourth quarter 2024.

Full Year 2025
•

•

Net sales increased 12% to $716.9 billion in 2025, compared with $638.0 billion in 2024. Excluding the $4.4 billion
favorable impact from year-over-year changes in foreign exchange rates throughout the year, net sales increased 12%
compared with 2024.
•

North America segment sales increased 10% year-over-year to $426.3 billion.

•

International segment sales increased 13% year-over-year to $161.9 billion, or increased 10% excluding
changes in foreign exchange rates.

•

AWS segment sales increased 20% year-over-year to $128.7 billion.

Operating income increased to $80.0 billion in 2025, compared with $68.6 billion in 2024.
•

North America segment operating income was $29.6 billion, compared with operating income of $25.0 billion
in 2024.

•

Source: C:\U

In [7]:
from langchain_groq import ChatGroq
import os

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",  # example Groq model
    temperature=0.7
)

response = llm.invoke("")
print(response.content)

It seems like you forgot to ask a question. What would you like to talk about? I can provide information on a wide range of topics.


In [8]:
import os
from dotenv import load_dotenv

from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_openai import ChatOpenAI, OpenAIEmbeddings


load_dotenv()


retriever = db.as_retriever(search_kwargs={"k": 3})


# ----------------------------
#  Contextualize Question
# ----------------------------

contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given a chat history and the latest user question "
     "which might reference context in the chat history, "
     "formulate a standalone question. Do NOT answer it."),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

contextualize_chain = (
    contextualize_prompt
    | llm
    | StrOutputParser()
)

# ----------------------------
# QA Prompt
# ----------------------------

qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering tasks. "
     "Use the following context to answer the question. "
     "If unknown, say you don't know. "
     "Use three sentences maximum.\n\n{context}"
    ),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

# ----------------------------
#  RAG Pipeline (LCEL)
# ----------------------------

rag_chain = (
    {
        "question": contextualize_chain,
        "history": RunnablePassthrough(),
    }
    | {
        "context": lambda x: retriever.invoke(x["question"]),
        "question": lambda x: x["question"],
        "history": lambda x: x["history"],
    }
    | qa_prompt
    | llm
    | StrOutputParser()
)

# ----------------------------
#  Add Message History
# ----------------------------

store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

# ----------------------------
#  Chat Loop
# ----------------------------

def continual_chat():
    print("Start chatting with the AI! Type 'exit' to end.")

    while True:
        query = input("You: ")
        if query.lower() == "exit":
            break

        response = conversational_rag.invoke(
            {"question": query},
            config={"configurable": {"session_id": "user1"}}
        )

        print("AI:", response)


if __name__ == "__main__":
    continual_chat()

Start chatting with the AI! Type 'exit' to end.
AI: I am reading an Amazon.com, Inc. earnings release document from Q4 2025.
AI: The document appears to be a financial report with detailed balance sheets, revenue data, and employee statistics.
